# Suite CNT — The container layout

`Container` gives each container shape a grid of cells, and places the children it gains and closes the gaps it loses, one change at a time, with no coordinates of any kind: mbse-basis [CAP-238: container schemas and grid insertion](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-238), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
from mbse.Schemas.Framework import Builders, Validators
from mbse.Diagrams.Framework import Container, Logical

NAMES = ["Container.Layout", "Container.OfGrid", "Container.Cells", "Container.OfCell", "Container.Holds",
         "Logical.OfShape"]
store = Builders.Store()
store.transaction(Logical.install)
store.transaction(Container.install)
L = store.Logical
diagrams = store.singleton("Logical.Diagrams")
problems = Validators.Validate(store).Reachable

with store.transaction():
    L.DiagramDirectory(diagrams, lambda f: f.diagrams(lambda a: a.insert(lambda e: e.name("overview").diagram(lambda d: d))))
overview = diagrams.diagrams({"name": "overview"})


def box(name, children=""):
    """A top-level shape of the diagram, holding a child of each name in `children`."""
    def spec(s):
        for child in children:
            s.children(lambda k, child=child: k.insert(lambda e: e.name(child).child(lambda c: c)))
        return s
    L.OfDiagram(overview, lambda d: d.shapes(lambda s: s.insert(lambda e: e.name(name).shape(spec))))
    return overview.shapes({"name": name})


def add(shape, name):
    L.OfShape(shape, lambda s: s.children(lambda k: k.insert(lambda e: e.name(name).child(lambda c: c))))
    Container.added(shape.children({"name": name}))


def remove(shape, name):
    L.OfShape(shape, lambda s: s.children(lambda k: k.remove(lambda e: e.name(name))))
    Container.removed(shape)


def cells(shape):
    """Each child's name and cell, in order, and the grid's extents."""
    (entry,) = shape.grid
    grid = entry.grid
    return [(e.name.get(), Container.cell_of(e.child)) for e in shape.children], (grid.rows.get(), grid.columns.get())


def raises(message, block):
    try:
        with store.transaction():
            block()
    except ValueError as error:
        assert str(error) == message, str(error)
    else:
        raise AssertionError(f"expected ValueError: {message}")

<a id="CNT-01"></a>
## CNT-01 · The install, after Logical's, defines the container layout's schemas, each valid, and run again writes nothing

In [ ]:
schemas = [store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
store.transaction(Container.install)
assert [store.registered(name) for name in NAMES] == schemas
assert {"grid", "cell"} <= set(store.registered("Logical.OfShape").adjacencies)

<a id="CNT-02"></a>
## CNT-02 · A grid placed on a container holds the children it already has, in order, each in its own cell

In [ ]:
with store.transaction():
    cpu = box("cpu", "abc")
    grid = Container.layout(cpu, 2)
assert [e.grid for e in cpu.grid] == [grid] and [e.shape for e in grid.shape] == [cpu]
assert (grid.grows.get(), grid.span.get()) == ("rows", 2)
assert cells(cpu) == ([("a", (0, 0)), ("b", (0, 1)), ("c", (1, 0))], (2, 2))
assert Container.cell_of(cpu) is None and problems(L.OfShape, cpu) == []

<a id="CNT-03"></a>
## CNT-03 · A child added takes the next cell, and the grid's extents grow

In [ ]:
with store.transaction():
    add(cpu, "d")
assert cells(cpu) == ([("a", (0, 0)), ("b", (0, 1)), ("c", (1, 0)), ("d", (1, 1))], (2, 2))
with store.transaction():
    add(cpu, "e")
assert cells(cpu) == ([("a", (0, 0)), ("b", (0, 1)), ("c", (1, 0)), ("d", (1, 1)), ("e", (2, 0))], (3, 2))
with store.transaction():
    one = box("one")
    Container.layout(one, 3)
assert cells(one) == ([], (0, 0))
with store.transaction():
    add(one, "x")
assert cells(one) == ([("x", (0, 0))], (1, 1)) and problems(L.OfShape, cpu) == []

<a id="CNT-04"></a>
## CNT-04 · A grid that grows columns fills each column first

In [ ]:
with store.transaction():
    bus = box("bus", "abcde")
    Container.layout(bus, 2, "columns")
assert cells(bus) == ([("a", (0, 0)), ("b", (1, 0)), ("c", (0, 1)), ("d", (1, 1)), ("e", (0, 2))], (2, 3))

<a id="CNT-05"></a>
## CNT-05 · A child removed leaves a gap the children after it close, one cell back each

In [ ]:
with store.transaction():
    remove(cpu, "b")
assert cells(cpu) == ([("a", (0, 0)), ("c", (0, 1)), ("d", (1, 0)), ("e", (1, 1))], (2, 2))
with store.transaction():
    remove(cpu, "e")
assert cells(cpu) == ([("a", (0, 0)), ("c", (0, 1)), ("d", (1, 0))], (2, 2))
with store.transaction():
    remove(bus, "a")
    remove(bus, "d")
assert cells(bus) == ([("b", (0, 0)), ("c", (1, 0)), ("e", (0, 1))], (2, 2))
with store.transaction():
    remove(one, "x")
assert cells(one) == ([], (0, 0)) and problems(L.OfShape, cpu) == [] and problems(L.OfShape, bus) == []

<a id="CNT-06"></a>
## CNT-06 · What a grid refuses

In [ ]:
raises("a grid grows rows or columns, not 'diagonal'", lambda: Container.layout(box("diagonal"), 2, "diagonal"))
raises("a grid's span is a positive int, not 0", lambda: Container.layout(box("empty"), 0))
raises("a shape has one grid", lambda: Container.layout(cpu, 2))
raises("a container without a grid has no container layout: give it one, with layout", lambda: add(box("bare"), "y"))